In [1]:
#created by xihuanzhu at 9.27

In [11]:
import glob
import os
import time
import skimage.draw
import nibabel as nib
import cv2

import pydicom as dicom

# import pydicom as dicom
import numpy as np
import SimpleITK as sitk
from shapely.geometry.polygon import Polygon
import torch
from PIL import Image, ImageOps, ImageFilter
from DropBlock import DropBlock2D
from model.FPN import FPN

In [5]:
# train_slice = np.load( "./npy_data/train_slice.npy")
# train_mask = np.load( "./npy_data/train_mask.npy")
# test_slice = np.load( "./npy_data/test_slice.npy")
# test_mask = np.load( "./npy_data/test_mask.npy")
# print(train_slice.shape)
# print(train_mask.shape)
# print(test_slice.shape)
# print(test_mask.shape)

In [4]:
# import matplotlib.pyplot as plt
# plt.imshow(np.squeeze(test_slice[0]), cmap ='gray')
# print(test_slice[0])

In [5]:
import random
SEED = 42

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device_ids=range(torch.cuda.device_count())
device = torch.device("cuda:0")

In [6]:
import albumentations as A
# from custom_data_aug import *

def Get_Train_Transform():
#     return A.Compose([
#         RandomHorizontalFlip(),
#         RandomRotate(degree=30),
#         RandomGaussianBlur()
#     ])
   return MyTransform(degree=30)

class MyTransform(object):
    def __init__(self, degree):
        self.degree = degree

    def __call__(self, sample):

        #print("&&&&&&&&&", sample['image'].dtype, "*****", sample['image'].shape, "%%%%%%%%%")
        img =  np.squeeze(sample['image']).astype(np.uint8)
        mask = np.squeeze(sample['label']).astype(np.uint8)
        img = Image.fromarray(img)
        mask = Image.fromarray(mask)
        # # print(img.type)

        #RandomRotate
        rotate_degree = random.uniform(-1*self.degree, self.degree)
        img = img.rotate(rotate_degree, Image.BILINEAR)
        mask = mask.rotate(rotate_degree, Image.NEAREST)
        
        #RandomHorizontalFlip
        if random.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
            
        #RandomGaussianBlur
        if random.random() < 0.5:
            img = img.filter(ImageFilter.GaussianBlur(
                radius=random.random()))

        img = np.array(img)
        mask = np.array(mask)
        img = img[np.newaxis, :, :]
        mask = mask[np.newaxis, :, :]

        return {'image': img, 'label': mask}

In [7]:
from albumentations import (
    PadIfNeeded,
    HorizontalFlip,
    VerticalFlip,    
    CenterCrop,    
    Crop,
    Compose,
    Transpose,
    RandomRotate90,
    ElasticTransform,
    GridDistortion, 
    OpticalDistortion,
    RandomSizedCrop,
    OneOf,
    CLAHE,
    RandomContrast,
    RandomGamma,
    RandomBrightness
)
def  augment_flips_color():
    return Compose([
        RandomSizedCrop(min_max_height=(512, 800), 
                           height=512, 
                           width=512, p=0.5),    
        VerticalFlip(p=0.5),              
        RandomRotate90(p=0.5),
        OneOf([ElasticTransform(p=0.5, 
                            alpha=120, 
                            sigma=120 * 0.05, 
                            alpha_affine=120 * 0.03),
        GridDistortion(p=0.5),
        OpticalDistortion(p=1, distort_limit=2, shift_limit=0.5)           
        ], p=0.8),
        CLAHE(p=0.8),
        RandomContrast(p=0.8),
        RandomBrightness(p=0.8),
        RandomGamma(p=0.8)])

aug = augment_flips_color()

In [8]:
from torch.utils.data import Dataset,DataLoader
from PIL import Image 
class DatasetRetriever(Dataset):
    def __init__(self, slice_path=None, mask_path=None, transforms=None):
        super().__init__()
        self.slice_path = slice_path
        self.mask_path = mask_path
        self.transforms = transforms
           
    def __getitem__(self, index):
        #print("***********************", index, "************************")
        image = np.load(f'{self.slice_path}/{str(index)}.npy')
        mask = np.load(f'{self.mask_path}/{str(index)}.npy')
        image = np.clip(image, -1000, 1000)
        sample = {'image': image, 'label': mask}
        #sample = self.transforms(sample)
        return sample['image'], sample['label']
    
    def __len__(self):
        return len(os.listdir(self.slice_path))
    

In [9]:
import torch

transforms = aug
batch_size = 12
num_workers=2
train_slice_path = "/raid/zhuxihuan/data/body/MC_uncertainty_single_data/train_slice"
train_mask_path = "/raid/zhuxihuan/data/body/MC_uncertainty_single_data/train_mask"
test_slice_path = "/raid/zhuxihuan/data/body/single_npy_data/val_slice"
test_mask_path = "/raid/zhuxihuan/data/body/single_npy_data/val_mask"

train_dataset = DatasetRetriever(slice_path=train_slice_path, mask_path=train_mask_path, transforms=transforms)
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size = batch_size,
    pin_memory=True,
    drop_last=True,
    num_workers=num_workers
)
test_dataset = DatasetRetriever(slice_path=test_slice_path, mask_path=test_mask_path, transforms=transforms)
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=4,
    pin_memory=True,
    drop_last=True,
    num_workers=1
)

In [10]:
print(len(test_loader))
print(len(train_loader))

345
464


In [11]:
import torch.nn as nn
import torch.nn.functional as F
class SoftDiceLoss(nn.Module):
    def __init__(self, weight=None, size_average=True):
        super(SoftDiceLoss, self).__init__()
        
    def forward(self, logits, targets):
        num = targets.size(0)
        smooth = 0.000001
        
        probs = torch.sigmoid(logits)
        m1 = probs.view(num, -1)
        m2 = targets.view(num, -1)
        intersection = (m1 * m2)
        
        score = 2. * (intersection.sum(1)) / (m1.sum(1) + m2.sum(1) + smooth)
        score = 1 - score.sum() / num
        return score

In [12]:
class Two_SoftDiceLoss(nn.Module):
    def __init__(self, weight=None, size_average=True):
        super(Two_SoftDiceLoss, self).__init__()
        
    def forward(self, logits, targets):
        num = targets.size(0)
        smooth = 0.000001
        
        probs = torch.sigmoid(logits)
        zero_probs  = 1 - probs
        zero_targets = 1 - targets
        
        m1 = probs.view(num, -1)
        m2 = targets.view(num, -1)
        intersection = (m1 * m2)
        score = 2. * (intersection.sum(1)) / (m1.sum(1) + m2.sum(1) + smooth)
        

        zero_m1 = zero_probs.view(num, -1)
        zero_m2 = zero_targets.view(num, -1)
        zero_intersection = (zero_m1 * zero_m2)
        zero_score = 2. * (zero_intersection.sum(1)) / (zero_m1.sum(1) + zero_m2.sum(1) + smooth)
        
        score = 2 - (score.sum() + zero_score.sum())/ num
        return score
    
class MyBCELoss(nn.Module):
    def __init__(self, weight=None, size_average=True):
        super(MyBCELoss, self).__init__()
        
    def forward(self, logits, targets):
        
        #probs = torch.softmax(logits)
        #probs = torch.sigmoid(logits)
        
        return torch.nn.BCEWithLogitsLoss()(logits, targets)
    
class DiceBCELoss(nn.Module):
    def __init__(self, weight=None, size_average=True):
        super(DiceBCELoss, self).__init__()
        
    def forward(self, logits, targets):
        return 0.5 * MyBCELoss()(logits, targets) + 0.5 * Two_SoftDiceLoss()(logits, targets)

In [13]:
  
""" Parts of the U-Net model """

import torch
import torch.nn as nn
import torch.nn.functional as F


class DoubleConv(nn.Module):
    """(convolution => [BN] => ReLU) * 2"""

    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):
    """Downscaling with maxpool then double conv"""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)


class Up(nn.Module):
    """Upscaling then double conv"""

    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()

        # if bilinear, use the normal convolutions to reduce the number of channels
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels , in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)


    def forward(self, x1, x2):
        x1 = self.up(x1)
        # input is CHW
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        # if you have padding issues, see
        # https://github.com/HaiyongJiang/U-Net-Pytorch-Unstructured-Buggy/commit/0e854509c2cea854e247a9c615f175f76fbb2e3a
        # https://github.com/xiaopeng-liao/Pytorch-UNet/commit/8ebac70e633bac59fc22bb5195e513d5832fb3bd
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)

""" Full assembly of the parts to form the complete network """

import torch.nn.functional as F

# class UNet(nn.Module):
#     def __init__(self, n_channels, n_classes, bilinear=True):
#         super(UNet, self).__init__()
#         self.n_channels = n_channels
#         self.n_classes = n_classes
#         self.bilinear = bilinear
#         self.drop = nn.Dropout(0.1)

#         self.inc = DoubleConv(n_channels, 64)
#         self.down1 = Down(64, 128)
#         self.down2 = Down(128, 256)
#         factor = 2 if bilinear else 1
#         #self.down3 = Down(256, 512 // factor)
#         self.down3 = Down(256, 512)
#         #factor = 2 if bilinear else 1
#         self.down4 = Down(512, 1024 // factor)
#         self.up1 = Up(1024, 512 // factor, bilinear)
#         self.up2 = Up(512, 256 // factor, bilinear)
#         self.up3 = Up(256, 128 // factor, bilinear)
#         self.up4 = Up(128, 64, bilinear)
#         self.outc = OutConv(64, n_classes)

#     def forward(self, x):
#         x1 = self.inc(x)
#         x2 = self.down1(x1)
#         x3 = self.down2(x2)
#         x3 = self.drop(x3)
#         x4 = self.down3(x3)
#         x5 = self.down4(x4)
#         x = self.up1(x5, x4)
#         x = self.up2(x, x3)
#         x = self.drop(x)
#         x = self.up3(x, x2)
#         x = self.up4(x, x1)
#         logits = self.outc(x)
#         return logits

class UNet(nn.Module):
    def __init__(self, n_channels, n_classes, bilinear=True):
        super(UNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear
        self.conv2_drop = DropBlock2D(0.88)

        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        factor = 2 if bilinear else 1
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = OutConv(64, n_classes)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x2 = self.conv2_drop(x2)
        x3 = self.down2(x2)
        x3 = self.conv2_drop(x3)
        x4 = self.down3(x3)
        x4 = self.conv2_drop(x4)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.conv2_drop(x)
        x = self.up2(x, x3)
        x = self.conv2_drop(x)
        x = self.up3(x, x2)
        x = self.conv2_drop(x)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits   
    
# netd = UNet(n_channels=1, n_classes=2).cuda()
# b = torch.tensor(torch.rand(3, 1, 512, 512)).to("cuda:0")
# a= netd(b)
# print(a.shape)

In [14]:
class TrainGlobalConfig:
    num_workeres = 2
    batch_size = 4
    n_epochs = 70
    lr = 3e-4
    base_dir = "./85_aug_dropblock_DiceBCE_class_model"
    if not os.path.exists(base_dir):
        os.makedirs(base_dir)
    log_path = f'{base_dir}/log.txt'
#     if os.path.exists(log_path):
#         os.remove(log_path)
    
    SchedulerClass = torch.optim.lr_scheduler.ReduceLROnPlateau
    scheduler_params = dict(
        mode='min',
        factor=0.8,
        patience=2,
        verbose=False,
        threshold=1e-4,
        threshold_mode='abs',
        cooldown=0,
        min_lr=1e-8,
        eps=1e-8
    )

In [15]:
from torch import optim
from glob import glob
import time
class Model_Pipeline(object):
    def __init__(self, model, device, config):
        self.config = config
        self.epoch = 0 
        self.best_loss = 10**5
        self.model = model
        self.device = device
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=config.lr, weight_decay=0.01)
        self.scheduler = config.SchedulerClass(self.optimizer, **config.scheduler_params)
        self.criterion = DiceBCELoss().to(self.device)
        self.log(f'Pipeline prepared. Device is {self.device}')
        
    def fit(self, train_loader, test_loader):
        for e in range(self.config.n_epochs):
            loss = self.train_one_epoch(train_loader)
            self.log(f'[RESULT]: Train. Epoch: {self.epoch}, loss: {loss:.5f}')
            print(f'[RESULT]: Train. Epoch: {self.epoch}, loss: {loss:.5f}')  
            
            loss = self.val_one_epoch(test_loader)
            self.log(f'[RESULT]: Test. Epoch: {self.epoch}, loss: {loss:.5f}')
            print(f'[RESULT]: Test epoch: {self.epoch}, loss: {loss:.5f}') 
                
            if loss < self.best_loss:
                self.best_loss = loss
                self.save_model(f'{self.config.base_dir}/best-loss-{str(self.epoch).zfill(3)}epoch.bin')
                for path in sorted(glob(f'{self.config.base_dir}/best-loss-*epoch.bin'))[:-6]:
                    os.remove(path)
                        
            # scheduler todo
            self.epoch += 1
                
    def train_one_epoch(self, train_loader):
        self.model.train()
        summary_loss = 0.0
            
        #print(list(enumerate(train_loader)))
        for step, (images, labels) in enumerate(train_loader):
            print(f'Train Step {step}/{len(train_loader)}, ' + f'summary_loss: {summary_loss:.5f}', end='\r')
            labels = labels.to(self.device).float()
            images = images.to(self.device).float()
                
            self.optimizer.zero_grad()
            output = self.model(images)
            #print(output.shape)
            loss = self.criterion(output, labels)
            summary_loss += loss
            loss.backward()
                
            self.optimizer.step()
                
        return summary_loss/len(train_loader)
        
    def val_one_epoch(self, test_loader):
        self.model.eval()
        summary_loss = 0.0
            
        for step, (images, labels) in enumerate(test_loader):
            with torch.no_grad():
                print(f'Test Step {step}/{len(test_loader)}, ' + f'summary_loss: {summary_loss:.5f}', end='\r')
                labels = labels.to(self.device).float()
                images = images.to(self.device).float()
                output = self.model(images)
                loss = self.criterion(output, labels)
                summary_loss += loss
                    
        return summary_loss/len(test_loader)
        
    def log(self, message):
        with open(self.config.log_path, 'a+') as logger:
            logger.write(f'{message}\n')
                
    def save_model(self, path):
        self.model.eval()
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'best_score': self.best_loss
         }, path)
    

In [16]:
def main():
    input_channels = 1
    output_channels = 1
    model = UNet(input_channels, output_channels).cuda()
    filtter = Model_Pipeline(model=model, device=device, config=TrainGlobalConfig)
    filtter.fit(train_loader, test_loader)
#main()

In [17]:

"""
----------------------------------------------
TEST
----------------------------------------------
"""

'\n----------------------------------------------\nTEST\n----------------------------------------------\n'

In [18]:
# class SoftDice(nn.Module):
#     def __init__(self, weight=None, size_average=True):
#         super(SoftDice, self).__init__()
        
#     def forward(self, logits, targets):
#         num = targets.size(0)
#         smooth = 0.000001
        
#         probs = F.sigmoid(logits)
#         m1 = probs.view(num, -1)
#         m2 = targets.view(num, -1)
#         intersection = (m1 * m2)
        
#         score = 2. * (intersection.sum(1)) / (m1.sum(1) + m2.sum(1) + smooth)
#         score = score.sum() / num
#         return score
class CEToSoftDice(nn.Module):
    def __init__(self, weight=None, size_average=True):
        super(CEToSoftDice, self).__init__()
        
    def forward(self, logits, targets):
        #logits = torch.nn.functional.softmax(torch.tensor(A), dim=-3)
        num = targets.size(0)
        smooth = 0.000001
        
        probs = F.sigmoid(logits)
        m1 = probs.view(num, -1)
        m2 = targets.view(num, -1)
        intersection = (m1 * m2)
        
        score = 2. * (intersection.sum(1)) / (m1.sum(1) + m2.sum(1) + smooth)
        score = score.sum() / num
        return score

In [19]:
# os.environ["CUDA_VISIBLE_DEVICES"] = "2"
# device_ids=range(torch.cuda.device_count())
# device = torch.device("cuda:0")
import torch
class Model_Predict(object):
    def __init__(self, model, device, log_path):
        self.model = model
        self.device = device
        self.criterion = CEToSoftDice().to(self.device)
        self.loss = 0
        self.log_path = log_path
        if os.path.exists(log_path):
            os.remove(log_path)
        #self.mask = mask
        pass
    
    def val_one_epoch(self, i, test_loader):
        self.model.train()
        summary_loss = 0
        mask = torch.Tensor(np.array([])).to(self.device)
#         img = torch.Tensor(np.array([])).to(self.device)
        #t_mask = torch.Tensor(np.array([])).to(self.device)
            
        for step, (images, labels) in enumerate(test_loader):
            with torch.no_grad():
                labels = labels.to(self.device).float()
                images = images.to(self.device).float()
                output = self.model(images)
#                 print(output.shape)
                if len(mask):
                    mask = torch.cat((mask, output))
                else:
                    mask = output
        
                loss = self.criterion(output, labels)
                self.log(f"A single two-dimensional picture'dice is {loss}")
                summary_loss += loss
        self.log(f"A patient'dice is {summary_loss/len(test_loader)}") 
        
        return summary_loss, mask
    
    def log(self, message):
        with open(self.log_path, 'a+') as logger:
            logger.write(f'{message}\n')

def load(path):
    checkpoint = torch.load(path)
    model = UNet(1, 1)
    model.load_state_dict(checkpoint['model_state_dict']) 
    return model

In [20]:
from torch.utils.data import Dataset,DataLoader
class DatasetRetrieverForOne(Dataset):
    def __init__(self, data_slice, data_mask, transforms=None):
        super().__init__()
        self.slicer = data_slice
        self.labels = data_mask
        self.transforms = transforms
           
    def __getitem__(self, index):
        """
        code for transform
        """
        return np.clip(self.slicer[index], -1000, 1000), self.labels[index]
    
    def __len__(self):
        return len(self.slicer)
    

In [21]:
train = [58, 24, 59, 30, 89, 
 54, 21, 1, 19, 29, 
 79, 15, 11, 26, 28,
 38, 3, 27, 91, 33, 
 75, 48, 44, 90, 82,
 43, 20, 7, 67, 42,
 80, 36, 53, 84, 72,
 10, 51, 25, 17, 60,
 73, 86, 83, 23, 62, 
 68, 50, 22]
val = [8, 66, 64, 61, 6, 47, 39, 37, 34, 18, 16]
val_path = "/raid/zhuxihuan/data/body/val_npy_data/"
print(os.listdir(val_path))
path = "/raid/zhuxihuan/data/body/test2_npy_data/"
test = np.array([])
for sdir in os.listdir(path):
    #print(sdir)
    #print(sdir[5:])
    #if int(sdir[5:]) in train or int(sdir[5:]) in val:
    if int(sdir[5:]) in train:
        #print("llll")
        pass
    else:
        test = np.append(test, sdir)
print(test)

['body_37', 'body_34', 'body_16', 'body_8', 'body_64', 'body_61', 'body_39', 'body_47', 'body_66', 'body_18', 'body_6']
['body_69' 'body_13' 'body_37' 'body_0' 'body_77' 'body_31' 'body_32'
 'body_5' 'body_45' 'body_40' 'body_78' 'body_92' 'body_81' 'body_71'
 'body_12' 'body_4' 'body_70' 'body_87' 'body_9' 'body_88' 'body_34'
 'body_16' 'body_8' 'body_64' 'body_61' 'body_39' 'body_47' 'body_66'
 'body_18' 'body_6' 'body_55' 'body_14' 'body_65' 'body_2']


In [22]:
import torch
#from npy2dcm import write_dicom

transforms = None
num_workers=2
sample = 4

#真实每个患者npy数据存储位置
t_val_dir = "/raid/zhuxihuan/data/body/test2_npy_data/"
#存储预测的每个患者的mask的npy目录
pre_val_dir = "/raid/zhuxihuan/data/body/dropblock_reg_train_dir"
if not os.path.exists(pre_val_dir):
    os.makedirs(pre_val_dir)

#存储预测的每个患者的mask的dcm文件目录
save_pre_dicom = "/raid/zhuxihuan/data/body/two_pre_test5_for_val_dcm_dir"
if not os.path.exists(save_pre_dicom):
    os.makedirs(save_pre_dicom)
    
def Get_Test_Result(sample, val_dir, pre_val_dir):   
    for sdir in os.listdir(t_val_dir)[:]:
        if sdir in test:
            f_dir = os.path.join(t_val_dir, sdir)
            t_slice = np.load(f_dir + "/slice.npy")
            t_mask = np.load(f_dir + "/mask.npy")
            print(t_slice.shape)

            test_dataset = DatasetRetrieverForOne(data_slice=t_slice, data_mask=t_mask, transforms=transforms)
            test_loader = torch.utils.data.DataLoader(
                test_dataset,
                batch_size=1,
                pin_memory=True,
                drop_last=True,
                num_workers=1
            )

            base_path = os.path.join(pre_val_dir, sdir)
            if not os.path.exists(base_path):
                os.makedirs(base_path)
            dcm_base_path = os.path.join(save_pre_dicom, sdir)
            if not os.path.exists(dcm_base_path):
                os.makedirs(dcm_base_path)
            for j in range(1, 5):    
                path = sorted(glob(f'85_aug_dropblock_DiceBCE_class_model/best-loss-*epoch.bin'))[-j]
                net = load(path).cuda()
                for i in range(sample):
                    print(f"/log{i + (j - 1) * sample}.txt")
                    pre = Model_Predict(model=net, device=device, log_path=base_path+f"/log{i + (j - 1) * sample}.txt")
                    _, pre_mask= pre.val_one_epoch(i, test_loader)

                    pre_mask = torch.sigmoid(pre_mask)
                    pre_mask = pre_mask.cpu()

                    np.save(base_path + f"/pmask{i + (j - 1) * sample}.npy", pre_mask)
                    print(pre_mask.dtype)
                    pre_mask = pre_mask.numpy().transpose(0, 2, 3, 1).astype(np.uint8)
                    print(pre_mask.shape, "*****")

           # np.save(base_path + f"/origin_mask{i}.npy", pre_mask)
            t_slice = np.squeeze(t_slice)
            print(t_slice.shape)
        
        #write_dicom(t_slice, pre_mask, dcm_base_path, roi_names=['body'], patient_id=sdir, pixel_spacing=(1.0, 1.0, 1.0))
        #print(pre_mask.shape)
        #return pre_mask
pre_mk = np.array([])       
Get_Test_Result(sample, t_val_dir, pre_val_dir)
    #np.append(pre_mk, temp)

#np.save(base_path + "/pmask.npy", pre_mask)

# for image, label in train_loader:
#         print(image.shape)
# print("*****")
# for image, label in test_loader:
#         print(image.shape)

(107, 1, 512, 512)
/log0.txt


/usr/local/anaconda3/lib/python3.6/site-packages/torch/nn/functional.py:1569: UserWarning: nn.functional.sigmoid is deprecated. Use torch.sigmoid instead.
  warnings.warn("nn.functional.sigmoid is deprecated. Use torch.sigmoid instead.")


torch.float32
(107, 512, 512, 1) *****
/log1.txt
torch.float32
(107, 512, 512, 1) *****
/log2.txt
torch.float32
(107, 512, 512, 1) *****
/log3.txt
torch.float32
(107, 512, 512, 1) *****
/log4.txt
torch.float32
(107, 512, 512, 1) *****
/log5.txt
torch.float32
(107, 512, 512, 1) *****
/log6.txt
torch.float32
(107, 512, 512, 1) *****
/log7.txt
torch.float32
(107, 512, 512, 1) *****
/log8.txt
torch.float32
(107, 512, 512, 1) *****
/log9.txt
torch.float32
(107, 512, 512, 1) *****
/log10.txt
torch.float32
(107, 512, 512, 1) *****
/log11.txt
torch.float32
(107, 512, 512, 1) *****
/log12.txt
torch.float32
(107, 512, 512, 1) *****
/log13.txt
torch.float32
(107, 512, 512, 1) *****
/log14.txt
torch.float32
(107, 512, 512, 1) *****
/log15.txt
torch.float32
(107, 512, 512, 1) *****
(107, 512, 512)
(125, 1, 512, 512)
/log0.txt
torch.float32
(125, 512, 512, 1) *****
/log1.txt
torch.float32
(125, 512, 512, 1) *****
/log2.txt
torch.float32
(125, 512, 512, 1) *****
/log3.txt
torch.float32
(125, 512, 512

/log0.txt
torch.float32
(80, 512, 512, 1) *****
/log1.txt
torch.float32
(80, 512, 512, 1) *****
/log2.txt
torch.float32
(80, 512, 512, 1) *****
/log3.txt
torch.float32
(80, 512, 512, 1) *****
/log4.txt
torch.float32
(80, 512, 512, 1) *****
/log5.txt
torch.float32
(80, 512, 512, 1) *****
/log6.txt
torch.float32
(80, 512, 512, 1) *****
/log7.txt
torch.float32
(80, 512, 512, 1) *****
/log8.txt
torch.float32
(80, 512, 512, 1) *****
/log9.txt
torch.float32
(80, 512, 512, 1) *****
/log10.txt
torch.float32
(80, 512, 512, 1) *****
/log11.txt
torch.float32
(80, 512, 512, 1) *****
/log12.txt
torch.float32
(80, 512, 512, 1) *****
/log13.txt
torch.float32
(80, 512, 512, 1) *****
/log14.txt
torch.float32
(80, 512, 512, 1) *****
/log15.txt
torch.float32
(80, 512, 512, 1) *****
(80, 512, 512)
(117, 1, 512, 512)
/log0.txt
torch.float32
(117, 512, 512, 1) *****
/log1.txt
torch.float32
(117, 512, 512, 1) *****
/log2.txt
torch.float32
(117, 512, 512, 1) *****
/log3.txt
torch.float32
(117, 512, 512, 1) **

/log0.txt
torch.float32
(95, 512, 512, 1) *****
/log1.txt
torch.float32
(95, 512, 512, 1) *****
/log2.txt
torch.float32
(95, 512, 512, 1) *****
/log3.txt
torch.float32
(95, 512, 512, 1) *****
/log4.txt
torch.float32
(95, 512, 512, 1) *****
/log5.txt
torch.float32
(95, 512, 512, 1) *****
/log6.txt
torch.float32
(95, 512, 512, 1) *****
/log7.txt
torch.float32
(95, 512, 512, 1) *****
/log8.txt
torch.float32
(95, 512, 512, 1) *****
/log9.txt
torch.float32
(95, 512, 512, 1) *****
/log10.txt
torch.float32
(95, 512, 512, 1) *****
/log11.txt
torch.float32
(95, 512, 512, 1) *****
/log12.txt
torch.float32
(95, 512, 512, 1) *****
/log13.txt
torch.float32
(95, 512, 512, 1) *****
/log14.txt
torch.float32
(95, 512, 512, 1) *****
/log15.txt
torch.float32
(95, 512, 512, 1) *****
(95, 512, 512)
(163, 1, 512, 512)
/log0.txt
torch.float32
(163, 512, 512, 1) *****
/log1.txt
torch.float32
(163, 512, 512, 1) *****
/log2.txt
torch.float32
(163, 512, 512, 1) *****
/log3.txt
torch.float32
(163, 512, 512, 1) **

(97, 1, 512, 512)
/log0.txt
torch.float32
(97, 512, 512, 1) *****
/log1.txt
torch.float32
(97, 512, 512, 1) *****
/log2.txt
torch.float32
(97, 512, 512, 1) *****
/log3.txt
torch.float32
(97, 512, 512, 1) *****
/log4.txt
torch.float32
(97, 512, 512, 1) *****
/log5.txt
torch.float32
(97, 512, 512, 1) *****
/log6.txt
torch.float32
(97, 512, 512, 1) *****
/log7.txt
torch.float32
(97, 512, 512, 1) *****
/log8.txt
torch.float32
(97, 512, 512, 1) *****
/log9.txt
torch.float32
(97, 512, 512, 1) *****
/log10.txt
torch.float32
(97, 512, 512, 1) *****
/log11.txt
torch.float32
(97, 512, 512, 1) *****
/log12.txt
torch.float32
(97, 512, 512, 1) *****
/log13.txt
torch.float32
(97, 512, 512, 1) *****
/log14.txt
torch.float32
(97, 512, 512, 1) *****
/log15.txt
torch.float32
(97, 512, 512, 1) *****
(97, 512, 512)
(116, 1, 512, 512)
/log0.txt
torch.float32
(116, 512, 512, 1) *****
/log1.txt
torch.float32
(116, 512, 512, 1) *****
/log2.txt
torch.float32
(116, 512, 512, 1) *****
/log3.txt
torch.float32
(11

In [24]:
file_dir = "/raid/zhuxihuan/data/body/dropblock_reg_train_dir/"
for sdir in os.listdir(file_dir):
    path = os.path.join(file_dir, sdir) + "/"
    print(path)
    pmk = []
    for s in range(sample * 4):
        temp = np.load(path + f"pmask{s}.npy")
        print(temp.shape)
        pmk.append(temp)
    print(len(pmk))
    mean_pmk = np.array(pmk)
    print(mean_pmk.shape)
    mean_pmk = mean_pmk.mean(axis=0)
    print(np.array(mean_pmk).shape)
    np.save(path + "mean_mask.npy", (mean_pmk))
    var_pmk = np.array(pmk)
    var_pmk = var_pmk.var(axis=0)
    print(var_pmk.shape)
    np.save(path + "var_mask.npy", var_pmk)

/raid/zhuxihuan/data/body/dropblock_reg_train_dir/body_69/
(107, 1, 512, 512)
(107, 1, 512, 512)
(107, 1, 512, 512)
(107, 1, 512, 512)
(107, 1, 512, 512)
(107, 1, 512, 512)
(107, 1, 512, 512)
(107, 1, 512, 512)
(107, 1, 512, 512)
(107, 1, 512, 512)
(107, 1, 512, 512)
(107, 1, 512, 512)
(107, 1, 512, 512)
(107, 1, 512, 512)
(107, 1, 512, 512)
(107, 1, 512, 512)
16
(16, 107, 1, 512, 512)
(107, 1, 512, 512)
(107, 1, 512, 512)
/raid/zhuxihuan/data/body/dropblock_reg_train_dir/body_13/
(125, 1, 512, 512)
(125, 1, 512, 512)
(125, 1, 512, 512)
(125, 1, 512, 512)
(125, 1, 512, 512)
(125, 1, 512, 512)
(125, 1, 512, 512)
(125, 1, 512, 512)
(125, 1, 512, 512)
(125, 1, 512, 512)
(125, 1, 512, 512)
(125, 1, 512, 512)
(125, 1, 512, 512)
(125, 1, 512, 512)
(125, 1, 512, 512)
(125, 1, 512, 512)
16
(16, 125, 1, 512, 512)
(125, 1, 512, 512)
(125, 1, 512, 512)
/raid/zhuxihuan/data/body/dropblock_reg_train_dir/body_37/
(111, 1, 512, 512)
(111, 1, 512, 512)
(111, 1, 512, 512)
(111, 1, 512, 512)
(111, 1, 51

(143, 1, 512, 512)
(143, 1, 512, 512)
(143, 1, 512, 512)
(143, 1, 512, 512)
(143, 1, 512, 512)
(143, 1, 512, 512)
(143, 1, 512, 512)
(143, 1, 512, 512)
16
(16, 143, 1, 512, 512)
(143, 1, 512, 512)
(143, 1, 512, 512)
/raid/zhuxihuan/data/body/dropblock_reg_train_dir/body_34/
(95, 1, 512, 512)
(95, 1, 512, 512)
(95, 1, 512, 512)
(95, 1, 512, 512)
(95, 1, 512, 512)
(95, 1, 512, 512)
(95, 1, 512, 512)
(95, 1, 512, 512)
(95, 1, 512, 512)
(95, 1, 512, 512)
(95, 1, 512, 512)
(95, 1, 512, 512)
(95, 1, 512, 512)
(95, 1, 512, 512)
(95, 1, 512, 512)
(95, 1, 512, 512)
16
(16, 95, 1, 512, 512)
(95, 1, 512, 512)
(95, 1, 512, 512)
/raid/zhuxihuan/data/body/dropblock_reg_train_dir/body_16/
(163, 1, 512, 512)
(163, 1, 512, 512)
(163, 1, 512, 512)
(163, 1, 512, 512)
(163, 1, 512, 512)
(163, 1, 512, 512)
(163, 1, 512, 512)
(163, 1, 512, 512)
(163, 1, 512, 512)
(163, 1, 512, 512)
(163, 1, 512, 512)
(163, 1, 512, 512)
(163, 1, 512, 512)
(163, 1, 512, 512)
(163, 1, 512, 512)
(163, 1, 512, 512)
16
(16, 163, 

In [ ]:
tets

hh = np.load("/raid/zhuxihuan/data/body/mc_dropout_BCE_pre_test5_dir/BodyV2_17-BodyV2_17/pmask0.npy")
print(hh.shape)

In [ ]:
A = np.random.randn(2, 2, 3,  3)
print(A)
c = torch.nn.functional.softmax(torch.tensor(A), dim=-3)
print(c)
k = torch.argmax(c, dim=-3)
print(k)

In [ ]:
def forward(logits, targets):
        num = targets.size(0)
        smooth = 0.000001
        
        probs = logits
        m1 = probs.view(num, -1)
        m2 = targets.view(num, -1)
        intersection = (m1 * m2)
        
        score = 2. * (intersection.sum(1)) / (m1.sum(1) + m2.sum(1) + smooth)
        score = score.sum() / num
        return score
A = np.array([[[[0.1,0.2],[0.5,0.4]]],
             [[[0.1,0.8],[0.6,0.4]]]])
C = np.array([[[[0.,0.],[1.,0.]]],
             [[[0.,1.],[1.,0.]]]])
B = np.array([[[[0,0],[0,0]]],
             [[[0,1],[0,0]]]])
print(forward(torch.tensor(A), torch.tensor(B)))
print(forward(torch.tensor(C), torch.tensor(B)))

In [ ]:
print(0.1*np.log(0.9))